# SageMaker Real-time Dynamic Batching Inference with Torchserve

---

This notebook's CI test result for us-west-2 is as follows. CI test results in other regions can be found at the end of the notebook. 

![This us-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-2/deploy_and_monitor|sm-batch_inference_with_torchserve|sm-batch_inference_with_torchserve.ipynb)

---

This notebook demonstrates the use of dynamic batching on SageMaker with [torchserve](https://github.com/pytorch/serve/) as a model server. It demonstrates the following
1. Batch inference using DLC i.e. SageMaker's default backend container. This is done using the SageMaker Python SDK v3 (sagemaker-core) with script-mode (a custom `inference.py` packed alongside a pre-trained model).
2. Specifying inference parameters for torchserve using environment variables.
3. Option to use a custom container with config file for torchserve baked-in the container.

**Imports**

In [1]:
# [exec-copy] pip install disabled; .v3test-venv already has local V3 editable install


In [2]:
import base64
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
import boto3, time, json

**Initiate session and retrieve region, account details**

In [3]:
# [exec-copy] explicit role/region injected for headless papermill run (no SageMaker metadata service).
from sagemaker.core.helper.session_helper import Session
import boto3
_boto_sess = boto3.Session(region_name="us-west-1")
sm_sess = Session(boto_session=_boto_sess)
role = "arn:aws:iam::729646638167:role/service-role/AmazonSageMaker-ExecutionRole-20251201T194045"


[07/16/26 10:36:17] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=5848373;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=5848374;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/lucasjia/Library/Application Support/sagemaker/config.yaml


In [4]:
sess = boto3.Session()
region = sess.region_name
account = boto3.client("sts").get_caller_identity().get("Account")

[07/16/26 10:36:18] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=5848379;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=5848380;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

**Prepare model**

In [5]:
bucket = sm_sess.default_bucket()
prefix = "ts-dynamic-batching"
default_bucket_prefix = sm_sess.default_bucket_prefix

# If a default bucket prefix is specified, append it to the s3 path
if default_bucket_prefix:
    prefix = f"{default_bucket_prefix}/{prefix}"

model_file_name = "BERTSeqClassification"

!aws s3 cp s3://torchserve/tar_gz_files/BERTSeqClassification.tar.gz .
!aws s3 cp BERTSeqClassification.tar.gz s3://{bucket}/{prefix}/models/

f"s3://{bucket}/{prefix}/models/"

download: s3://torchserve/tar_gz_files/BERTSeqClassification.tar.gz to ./BERTSeqClassification.tar.gz


upload: ./BERTSeqClassification.tar.gz to s3://sagemaker-us-west-1-729646638167/ts-dynamic-batching/models/BERTSeqClassification.tar.gz


's3://sagemaker-us-west-1-729646638167/ts-dynamic-batching/models/'

In [6]:
model_artifact = f"s3://{bucket}/{prefix}/models/{model_file_name}.tar.gz"

In [7]:
model_name = "hf-dynamic-torchserve-sagemaker"

## Use AWS Deep Learning Container

In [8]:
# V3: image_uris moved to sagemaker.core. We'll use a pytorch inference DLC image that ships with
# sagemaker-pytorch-inference-toolkit v2.0.6, which supports the Torchserve environment variables used below.
from sagemaker.core import image_uris

image_uri = image_uris.retrieve(
    framework="pytorch",
    region=region,
    py_version="py39",
    image_scope="inference",
    version="1.13.1",
    instance_type="ml.c5.9xlarge",
)

[07/16/26 10:36:37] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=5848385;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=5848386;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

In [9]:
image_uri

'763104351884.dkr.ecr.us-west-1.amazonaws.com/pytorch-inference:1.13.1-cpu-py39'

#### Create SageMaker model, deploy and predict

In V2 `PyTorchModel(source_dir="code", entry_point="inference.py")` automatically repacked the pre-trained `model.tar.gz` so that our custom `inference.py` (and its `requirements.txt`) were bundled under `code/`, and it set the `SAGEMAKER_PROGRAM` / `SAGEMAKER_SUBMIT_DIRECTORY` environment variables for script mode.

In V3 we do this explicitly:
1. `repack_model` (from `sagemaker.core.common_utils`) unpacks the pre-trained model tarball, injects our `code/` directory, and re-uploads a repacked `model.tar.gz` to S3.
2. `sagemaker.core.resources.Model` / `EndpointConfig` / `Endpoint` create the hosted endpoint. The Torchserve dynamic-batching parameters are passed as container environment variables, exactly as in V2, alongside the script-mode `SAGEMAKER_PROGRAM` / `SAGEMAKER_SUBMIT_DIRECTORY` vars.

In [10]:
# V3: repack the pre-trained model tarball to bundle our custom inference code under code/.
# This mirrors what V2's PyTorchModel(source_dir=..., entry_point=...) did automatically.
from sagemaker.core.common_utils import repack_model

repacked_model_artifact = f"s3://{bucket}/{prefix}/models/{model_file_name}-repacked.tar.gz"

repack_model(
    inference_script="inference.py",
    source_directory="code",
    dependencies=[],
    model_uri=model_artifact,
    repacked_model_uri=repacked_model_artifact,
    sagemaker_session=sm_sess,
)
repacked_model_artifact

's3://sagemaker-us-west-1-729646638167/ts-dynamic-batching/models/BERTSeqClassification-repacked.tar.gz'

In [11]:
# V3: create Model + EndpointConfig + Endpoint with sagemaker-core resources instead of
# PyTorchModel.deploy(). The Torchserve dynamic-batching env vars are preserved verbatim; the
# repacked model already contains code/inference.py, so we point script mode at /opt/ml/model/code.
import time
from sagemaker.core.resources import Model, EndpointConfig, Endpoint
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant

env_variables_dict = {
    "SAGEMAKER_TS_BATCH_SIZE": "3",
    "SAGEMAKER_TS_MAX_BATCH_DELAY": "100000",
    "SAGEMAKER_TS_MIN_WORKERS": "1",
    "SAGEMAKER_TS_MAX_WORKERS": "1",
    # script-mode vars (V2 set these automatically during repack):
    "SAGEMAKER_PROGRAM": "inference.py",
    "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
}

# Change the instance type as necessary.
instance_type = "ml.c5.18xlarge"

deploy_suffix = time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())
sm_model_name = f"{model_name}-{deploy_suffix}"
endpoint_config_name = f"{model_name}-epc-{deploy_suffix}"
endpoint_name = f"{model_name}-ep-{deploy_suffix}"

Model.create(
    model_name=sm_model_name,
    primary_container=ContainerDefinition(
        image=image_uri,
        model_data_url=repacked_model_artifact,
        environment=env_variables_dict,
    ),
    execution_role_arn=role,
)

EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=sm_model_name,
            initial_instance_count=1,
            instance_type=instance_type,
            initial_variant_weight=1.0,
        )
    ],
)

predictor = Endpoint.create(
    endpoint_name=endpoint_name,
    endpoint_config_name=endpoint_config_name,
)
predictor.wait_for_status("InService")

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/lucasjia/Library/Application Support/sagemaker/config.yaml


[07/16/26 10:36:59] INFO     Creating model resource.                                            ]8;id=5848393;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5848394;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#20593\20593]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=5848401;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=5848402;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py#361\361]8;;\

                    INFO     Runs on sagemaker prod, region:us-west-1                                  ]8;id=5848408;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=5848409;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py#375\375]8;;\

...

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=5848420;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=5848421;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

[07/16/26 10:37:00] INFO     Creating endpoint_config resource.                                  ]8;id=5848427;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5848428;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#11069\11069]8;;\

                    INFO     Creating endpoint resource.                                         ]8;id=5848434;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5848435;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#10228\10228]8;;\

Output()

[07/16/26 10:40:33] INFO     Final Resource Status: InService                                    ]8;id=5848441;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5848442;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#10484\10484]8;;\

## Predictions

#### By spawning a pool of 3 processes we're able to simulate requests from multiple clients and verify inference results

The endpoint is configured with `SAGEMAKER_TS_BATCH_SIZE=3` and a large `SAGEMAKER_TS_MAX_BATCH_DELAY`, so Torchserve waits until it has 3 concurrent requests before running a single batched inference. We therefore send 3 requests concurrently.

In [12]:
# V3: each worker builds its own sagemaker-core Endpoint handle and calls invoke().
# Endpoint.invoke replaces the V2 Predictor.predict; we JSON-serialize the input and read the
# response body ourselves (equivalent to the V2 JSONSerializer + BytesDeserializer pair).
import multiprocessing
from multiprocessing.pool import ThreadPool  # [exec-copy] avoid spawn/fork issues under papermill


def invoke(endpoint_name):
    from sagemaker.core.resources import Endpoint

    endpoint = Endpoint(endpoint_name=endpoint_name)
    response = endpoint.invoke(
        body=json.dumps(
            "{Bloomberg has decided to publish a new report on global economic situation.}"
        ),
        content_type="application/json",
        accept="application/json",
    )
    payload = response.body.read()
    if isinstance(payload, bytes):
        payload = payload.decode("utf-8")
    return payload


endpoint_name = predictor.endpoint_name
pool = ThreadPool(3)
results = pool.map(invoke, 3 * [endpoint_name])
pool.close()
pool.join()
print(results)

['["Not Accepted"]', '["Not Accepted"]', '["Not Accepted"]']


In [13]:
# V3: clean up the hosted endpoint, its config, and the model.
predictor.delete()
EndpointConfig.get(endpoint_config_name).delete()
Model.get(sm_model_name).delete()

                    INFO     Deleting Endpoint -                                                 ]8;id=5853448;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5853449;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#10428\10428]8;;\
                             hf-dynamic-torchserve-sagemaker-ep-2026-07-16-17-36-59                                

                    INFO     Deleting EndpointConfig -                                           ]8;id=5853455;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5853456;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#11220\11220]8;;\
                             hf-dynamic-torchserve-sagemaker-epc-2026-07-16-17-36-59                               

[07/16/26 10:40:36] INFO     Deleting Model -                                                    ]8;id=5853462;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5853463;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#20740\20740]8;;\
                             hf-dynamic-torchserve-sagemaker-2026-07-16-17-36-59                                   

## Conclusion

Through this exercise, we were able to understand the basics of batch inference using torchserve on Amazon SageMaker. We learnt that we can have several inference requests from different processes/users batched together, and the results will be processed as a batch of inputs. We also learnt that we could either use SageMaker's default DLC container as the base environment, and supply an inference.py script with the model, or create a custom container that can be used with SageMaker for more involved workflows.

## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.

![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/deploy_and_monitor|sm-batch_inference_with_torchserve|sm-batch_inference_with_torchserve.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-2/deploy_and_monitor|sm-batch_inference_with_torchserve|sm-batch_inference_with_torchserve.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/deploy_and_monitor|sm-batch_inference_with_torchserve|sm-batch_inference_with_torchserve.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ca-central-1/deploy_and_monitor|sm-batch_inference_with_torchserve|sm-batch_inference_with_torchserve.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/sa-east-1/deploy_and_monitor|sm-batch_inference_with_torchserve|sm-batch_inference_with_torchserve.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-1/deploy_and_monitor|sm-batch_inference_with_torchserve|sm-batch_inference_with_torchserve.ipynb)

![This eu-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-2/deploy_and_monitor|sm-batch_inference_with_torchserve|sm-batch_inference_with_torchserve.ipynb)

![This eu-west-3 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-3/deploy_and_monitor|sm-batch_inference_with_torchserve|sm-batch_inference_with_torchserve.ipynb)

![This eu-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-central-1/deploy_and_monitor|sm-batch_inference_with_torchserve|sm-batch_inference_with_torchserve.ipynb)

![This eu-north-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-north-1/deploy_and_monitor|sm-batch_inference_with_torchserve|sm-batch_inference_with_torchserve.ipynb)

![This ap-southeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-1/deploy_and_monitor|sm-batch_inference_with_torchserve|sm-batch_inference_with_torchserve.ipynb)

![This ap-southeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-2/deploy_and_monitor|sm-batch_inference_with_torchserve|sm-batch_inference_with_torchserve.ipynb)

![This ap-northeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-1/deploy_and_monitor|sm-batch_inference_with_torchserve|sm-batch_inference_with_torchserve.ipynb)

![This ap-northeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-2/deploy_and_monitor|sm-batch_inference_with_torchserve|sm-batch_inference_with_torchserve.ipynb)

![This ap-south-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-south-1/deploy_and_monitor|sm-batch_inference_with_torchserve|sm-batch_inference_with_torchserve.ipynb)
